# KIST 16S Full-Length · 03 PC/NC 및 지정 3개 제외
**Dataset:** H209SC26096091 · PacBio Revio HiFi · 메타데이터 110개 샘플 (PC 2, NC 2 포함)

이 노트북은 **PC/NC 4개를 제거하고 지정한 3개를 추가로 제거. 정확한 식별자가 입력되기 전에는 BLOCKED** 시나리오만 실행합니다.

1. Drive 인증 → Excel 샘플 ID 매핑 → 기존 ZIP 내 ASV 테이블 또는 FASTQ 기반 분석 준비
2. 선택한 단일 시나리오에 대해 ASV abundance, alpha/beta diversity, PCoA, genus plot, HTML 리포트 생성
3. Google Drive `analysis_outputs/full16s_no_controls_no_three_YYYYMMDD...`에 개별 저장

**주의:** ZIP은 결과 아카이브로, 원시 FASTQ 여부가 확인되지 않았습니다. 샘플 제외에 따른 지표 재산출은 공통 ASV 테이블을 재사용하며 DADA2 재학습을 의미하지 않습니다.

**필수:** 아래 `EXCLUDE_THREE_SAMPLE_IDS`에 *정확한 3개 샘플 이름* 또는 전체 샘플 ID를 입력하세요. 미입력 시 분석을 중단합니다.


In [ ]:
# 0. Inputs, toggles and exact exclusion IDs
DRIVE_PROJECT_ID = "1070GDz6WnKV23Ygzgs08idwO1p4-xYm5"
DRIVE_METADATA_XLSX_ID = "1_USCzdwWfQ614tjzDocbGDcM5aZp-YBD"
DRIVE_EXISTING_ZIP_ID = "1rPpGDmZHDcjNAm0UmjP1kZl5VdGB9478"
DRIVE_FASTQ_FOLDER_ID = "1ujVQlcJtzMRYEqGone8Dw_viN53IzjyB"
DRIVE_REPORT_FOLDER_ID = "1B_twbeB9W_m2piDNdxbyG7LYEkIMbRgv"
DRIVE_REFERENCE_FOLDER_ID = "1Ed4eWpq7kCTaJG0A81PZ42tfX2mtga7-"
GITHUB_REF = "feat/full-length-16s-hifi-pipeline-20261112"
RUN_SCENARIO = "no_controls_no_three"
DOWNLOAD_VENDOR_ZIP = True  # Set True if you want to inspect/re-use the 2.17 GB results archive.
DOWNLOAD_RAW_FASTQ = False  # Set True only after uploading demultiplexed HiFi FASTQ to raw_fastq.
RUN_NEW_DADA2 = False       # Requires verified primers and QIIME 2 install.
QIIME_CONDA_ENV = "rachis-qiime2-2026.10"
INSTALL_QIIME = False       # Heavy install; Colab RAM/disk and internet required.
FORWARD_PRIMER_5TO3 = ""    # Required! Verify laboratory primer sequence.
REVERSE_PRIMER_5TO3 = ""    # Optional; verify 5'->3' orientation before use.
EXCLUDE_THREE_SAMPLE_IDS = []  # REQUIRED: enter exactly three unique names/IDs seen in the Excel mapping.
from pathlib import Path
WORK = Path("/content/kist_full_length_16s/no_controls_no_three")
WORK.mkdir(exist_ok=True, parents=True)
print("Work directory:", WORK)

In [ ]:
# 1. Authenticate Drive and fetch Python code from GitHub
import os, sys, json, io, shutil, subprocess, requests, zipfile
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive", force_remount=False)
subprocess.run([sys.executable,"-m","pip","install","-q","pandas","numpy","scipy","matplotlib","openpyxl"],check=True)
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import google.auth, pandas as pd
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive_api = build("drive","v3",credentials=creds,cache_discovery=False)
def list_drive(folder):
    files, token = [], None
    while True:
        page = drive_api.files().list(q="'"+folder+"' in parents and trashed = false",
               fields="nextPageToken,files(id,name,size,mimeType)",page_size=1000,page_token=token).execute()
        files.extend(page.get("files",[]))
        token=page.get("nextPageToken")
        if not token: break
    return files
def download_from_drive(file_id, target):
    target = Path(target)
    if target.exists() and target.stat().st_size > 0:
        print("Cached:",target.name); return target
    target.parent.mkdir(parents=True,exist_ok=True)
    req=drive_api.files().get_media(fileId=file_id)
    with target.open("wb") as f:
        dl=MediaIoBaseDownload(f,req,chunksize=16*1024*1024)
        done=False
        while not done:
            _,done=dl.next_chunk()
    return target
def github_file(path):
    url="https://raw.githubusercontent.com/kimtk94/Codex/"+GITHUB_REF+"/full_length_16s/"+path
    response=requests.get(url,timeout=60); response.raise_for_status()
    result=WORK/path; result.parent.mkdir(parents=True,exist_ok=True)
    result.write_bytes(response.content)
    return result
MAIN=github_file("src/full16s.py")
SUBSETS=github_file("src/subset_analyses.py")
QIIME_SCRIPT=github_file("scripts/run_qiime2.sh")
print("GitHub files retrieved",MAIN.name,SUBSETS.name,QIIME_SCRIPT.name)

In [ ]:
# 2. Build exact metadata: repeated NC/PC names are preserved as distinct Novogene IDs
XLSX=download_from_drive(DRIVE_METADATA_XLSX_ID,WORK/"metadata/Pacbio_Analysis_Form.xlsx")
META=WORK/"metadata/sample_metadata.tsv"
subprocess.run([sys.executable,str(MAIN),"samples",str(XLSX),str(META)],check=True)
metadata=pd.read_csv(META,sep="\t",dtype=str).fillna("")
print("Total:",len(metadata),"; PC:",sum(metadata["control"]=="PC"),"; NC:",sum(metadata["control"]=="NC"))
display(metadata[["sample-id","sample-name","novogene-id","cohort","week","control"]])

In [ ]:
# 3. The three sample IDs in the image require exact confirmation; do not invent identities.
# You may paste either unique 'sample-name' values OR exact 'sample-id' strings.
EXCLUDE_THREE_SAMPLE_IDS=[x.strip() for x in EXCLUDE_THREE_SAMPLE_IDS if str(x).strip()]
EXCLUDE_FILE=None
if EXCLUDE_THREE_SAMPLE_IDS:
    converted=[]
    for value in EXCLUDE_THREE_SAMPLE_IDS:
        if value in set(metadata["sample-id"]):
            converted.append(value)
        else:
            matches=metadata.loc[metadata["sample-name"]==value,"sample-id"].tolist()
            if len(matches)!=1:
                raise ValueError("Ambiguous/unrecognized sample: "+value+"; use full sample-id")
            converted.append(matches[0])
    if len(converted)!=3 or len(set(converted))!=3:
        raise ValueError("Exactly THREE unique samples must be specified.")
    EXCLUDE_FILE=WORK/"metadata/exclude_three_samples.tsv"
    pd.DataFrame({"sample-id":converted}).to_csv(EXCLUDE_FILE,sep="\t",index=False)
    print("Validated three exclusions:",converted)
else:
    raise ValueError("BLOCKED: specify exactly three sample IDs in EXCLUDE_THREE_SAMPLE_IDS and rerun cell 3.")
print("Control exclusions:",metadata.loc[metadata["control"].isin(["PC","NC"]), "sample-id"].tolist())

In [ ]:
# 4. (Optional) Inspect vendor 2.17 GB ZIP and extract a candidate feature table.
ZIP=WORK/"vendor/Result_X209SC26096091-Z01-F003_16S.zip"
FEATURE=WORK/"qiime2_outputs/feature-table.tsv"
TAXONOMY=WORK/"qiime2_outputs/taxonomy_export/taxonomy.tsv"
if DOWNLOAD_VENDOR_ZIP:
    download_from_drive(DRIVE_EXISTING_ZIP_ID, ZIP)
    subprocess.run([sys.executable,str(MAIN),"inspect",str(ZIP),
                    str(WORK/"vendor/zip_inventory.json")],check=True)
    with zipfile.ZipFile(ZIP) as z:
        tabular=[x for x in z.namelist() if x.lower().endswith("feature-table.tsv") and not x.endswith("/")]
        tax=[x for x in z.namelist() if x.lower().endswith("taxonomy.tsv") and not x.endswith("/")]
        print("Candidate feature tables:",tabular[:15])
        if len(tabular)==1:
            FEATURE.parent.mkdir(parents=True,exist_ok=True)
            with z.open(tabular[0]) as source, FEATURE.open("wb") as target: shutil.copyfileobj(source,target)
        if len(tax)==1:
            TAXONOMY.parent.mkdir(parents=True,exist_ok=True)
            with z.open(tax[0]) as source, TAXONOMY.open("wb") as target: shutil.copyfileobj(source,target)
        if len(tabular)>1:
            print("BLOCKED: multiple candidate feature tables, inspect and select manually.")
else:
    print("Vendor ZIP download skipped; set DOWNLOAD_VENDOR_ZIP=True to inspect ~2.17 GB file.")

In [ ]:
# 5. (Optional) Download HiFi FASTQs, build manifest and check reads.
FASTQ_DIR=WORK/"raw_fastq"
MANIFEST=WORK/"metadata/manifest.tsv"
raw_files=[x for x in list_drive(DRIVE_FASTQ_FOLDER_ID) if x["name"].lower().endswith((".fastq.gz",".fq.gz",".fastq",".fq"))]
print("Raw FASTQ files in Drive:",len(raw_files))
if DOWNLOAD_RAW_FASTQ and raw_files:
    FASTQ_DIR.mkdir(parents=True,exist_ok=True)
    for item in raw_files:
        download_from_drive(item["id"],FASTQ_DIR/item["name"])
    subprocess.run([sys.executable,str(MAIN),"manifest",str(FASTQ_DIR),
                    str(META),str(MANIFEST),str(WORK/"prepared_fastq")],check=True)
    subprocess.run([sys.executable,str(MAIN),"qc",str(MANIFEST),
                    str(WORK/"metadata/fastq_qc.tsv")],check=True)
else:
    print("Raw sequencing workflow not executed: missing FASTQs or download flag disabled.")

In [ ]:
# 6. (Optional, heavy) install QIIME 2 2026.10 + denoise PacBio HiFi.
# Direct official environment: https://library.qiime2.org/quickstart/qiime2
# Colab may lack resources for the full environment. GitHub Actions only handles CI.
if INSTALL_QIIME or RUN_NEW_DADA2:
    import tarfile
    micromamba=WORK/"bin/micromamba"
    if not micromamba.exists():
        url="https://micro.mamba.pm/api/micromamba/linux-64/latest"
        res=requests.get(url,timeout=180);res.raise_for_status()
        with tarfile.open(fileobj=io.BytesIO(res.content),mode="r:bz2") as tar:
            member=next(item for item in tar.getmembers() if item.name=="bin/micromamba")
            micromamba.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(member) as src,micromamba.open("wb") as dst:shutil.copyfileobj(src,dst)
        micromamba.chmod(0o755)
    env={**os.environ,"MAMBA_ROOT_PREFIX":str(WORK/"micromamba")}
    yaml="https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.10/qiime2/released/rachis-qiime2-linux-64-conda.yml"
    if INSTALL_QIIME:
        subprocess.run([str(micromamba),"create","-y","-n",QIIME_CONDA_ENV,"-f",yaml],check=True,env=env)
    if RUN_NEW_DADA2:
        if not MANIFEST.exists(): raise FileNotFoundError("Upload FASTQs and run cell 5.")
        if not FORWARD_PRIMER_5TO3: raise ValueError("Required actual forward primer not configured.")
        candidate_qza=[x for x in list_drive(DRIVE_REFERENCE_FOLDER_ID) if x["name"].lower().endswith(".qza")]
        CLASSIFIER=""
        if candidate_qza:
            item=candidate_qza[0]
            CLASSIFIER=str(download_from_drive(item["id"],WORK/"references"/item["name"]))
        cmd=[str(micromamba),"run","-n",QIIME_CONDA_ENV,"bash",str(QIIME_SCRIPT),
            str(MANIFEST),str(FEATURE.parent),FORWARD_PRIMER_5TO3,CLASSIFIER,REVERSE_PRIMER_5TO3]
        subprocess.run(cmd,env=env,check=True)
else:
    print("QIIME 2 optional denoising not requested.")

In [ ]:
# 7. Analyze ONLY this notebook's selected scenario
OUTPUT = WORK / "reports"
if FEATURE.exists():
    cmd = [sys.executable, str(SUBSETS), "--feature-table", str(FEATURE),
           "--metadata", str(META), "--output-dir", str(OUTPUT),
           "--scenario", RUN_SCENARIO]
    if TAXONOMY.exists():
        cmd.extend(["--taxonomy", str(TAXONOMY)])
    if EXCLUDE_FILE:
        cmd.extend(["--exclude-three", str(EXCLUDE_FILE)])
    if MANIFEST.exists():
        cmd.extend(["--manifest", str(MANIFEST)])
    subprocess.run(cmd, check=True)
    scenario_summary = pd.read_csv(OUTPUT / "scenario_comparison.csv")
    display(scenario_summary)
    selected = scenario_summary.iloc[0]
    if selected["status"] != "PASS":
        raise RuntimeError("Scenario not completed: "+str(selected.to_dict()))
    print("Report:", OUTPUT / RUN_SCENARIO / "report.html")
else:
    raise FileNotFoundError("No valid ASV feature-table.tsv found. Inspect the vendor ZIP contents or supply HiFi FASTQ.")


In [ ]:
# 8. Upload all generated small outputs into a timestamped folder in the specified Drive directory.
from datetime import datetime,timezone
extensions={".html",".png",".tsv",".json",".csv"}
files=[]
for source in [WORK/"metadata",WORK/"vendor",WORK/"reports",WORK/"qiime2_outputs"]:
    if source.exists():
        files += [f for f in source.rglob("*") if f.is_file() and f.suffix.lower() in extensions]
if files:
    name="full16s_"+RUN_SCENARIO+"_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    folder=drive_api.files().create(body={"name":name,"mimeType":"application/vnd.google-apps.folder",
                  "parents":[DRIVE_REPORT_FOLDER_ID]},fields="id").execute()
    dest=folder["id"]
    for f in files:
        if f.stat().st_size>50*1024*1024:
            print("Skip big output:",f);continue
        relative=f.relative_to(WORK)
        # Prefix to preserve relative path; avoid overwriting same names across variants.
        filename=str(relative).replace("/","__")
        mime="image/png" if f.suffix==".png" else ("text/html" if f.suffix==".html" else "text/plain")
        upload=MediaFileUpload(str(f),mimetype=mime,resumable=True)
        drive_api.files().create(body={"name":filename,"parents":[dest]},
              media_body=upload,fields="id").execute()
    print("Drive results:", "https://drive.google.com/drive/folders/"+dest)
else:
    print("No output reports to upload; check blocked scenarios above.")